# PtA_IC Conversion

In [12]:
# Fix accessibility_summary_3000m parquets with correct municipality data
SWISS_CRS = "EPSG:2056"

grid_points_fixed = gpd.read_parquet("outputs/outputs_nearest_ic/grid_points_3000m.parquet")
if grid_points_fixed.crs is None:
    grid_points_fixed = grid_points_fixed.set_crs("EPSG:4326")
grid_points_fixed["id"] = grid_points_fixed["id"].astype(str)
muni_lookup = grid_points_fixed[["id", "GDENAME", "KTNAME", "KTKZ"]].set_index("id")

for departure in DEPARTURES:
    for minutes in MAX_TIMES:
        label = max_time_label(minutes)
        path = OUTPUTS_DIR / f"accessibility_summary_3000m_departure_{departure}_max_{label}.parquet"
        if not path.exists():
            continue

        df = gpd.read_parquet(path)
        df["id"] = df["id"].astype(str)

        # Drop stale municipality columns
        muni_cols = ["GDENAME", "KTNAME", "KTKZ", "GDEHISTID", "GDENR", "KTNR",
                     "GDENAME_left", "GDENAME_right", "KTNAME_left", "KTNAME_right",
                     "KTKZ_left", "KTKZ_right"]
        df = df.drop(columns=[c for c in muni_cols if c in df.columns], errors="ignore")

        # Re-attach from fixed grid_points
        df = df.join(muni_lookup, on="id", how="left")
        df.to_parquet(path, index=False)

print("Fixed all accessibility_summary_3000m parquets")

Fixed all accessibility_summary_3000m parquets


In [13]:
from pathlib import Path
import json

import geopandas as gpd
import pandas as pd

In [14]:
OUTPUTS_DIR = Path("outputs/outputs_nearest_ic")
DOCS_DATA_DIR = Path("docs/data")
PTA_IC_DOCS_DIR = DOCS_DATA_DIR / "ptA_IC"
PTA_PTB_OUTPUTS_DIR = Path("outputs/outputs_ptA_ptB")

DEPARTURES = ["0900", "1200", "1700", "2200"]
MAX_TIMES = [30, 60, 90, 120, 150, 180, 210, 240]

PTA_IC_DOCS_DIR.mkdir(parents=True, exist_ok=True)

def resolve_ptb_chunk_file(path):
    path = Path(path)
    if path.exists():
        return path
    return PTA_PTB_OUTPUTS_DIR / "chunks" / path.name

In [15]:
def max_time_label(minutes: int) -> str:
    return f"{minutes:03d}min"


def required_pta_ic_files():
    files = [OUTPUTS_DIR / "ic_stations.parquet"]
    for departure in DEPARTURES:
        for minutes in MAX_TIMES:
            label = max_time_label(minutes)
            files.append(OUTPUTS_DIR / f"accessibility_summary_3000m_departure_{departure}_max_{label}.parquet")
            files.append(OUTPUTS_DIR / f"od_pairs_3000m_departure_{departure}_max_{label}.parquet")
    return files


missing = [path for path in required_pta_ic_files() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required ptA_IC output files: {missing[:5]}")

print("All ptA_IC input files are present.")

All ptA_IC input files are present.


In [16]:
ic_hubs = gpd.read_parquet(OUTPUTS_DIR / "ic_stations.parquet").to_crs("EPSG:4326")
ic_lookup = pd.DataFrame(
    {
        "to_id": ic_hubs["id"].astype(str),
        "stop_name": ic_hubs["stop_name"],
        "lon": ic_hubs.geometry.x,
        "lat": ic_hubs.geometry.y,
    }
)

manifest = {
    "departures": [{"label": f"{d[:2]}:{d[2:]}", "value": d} for d in DEPARTURES],
    "max_times": MAX_TIMES,
    "default_departure": "0900",
    "default_max_time": 30,
    "files": {},
}

for departure in DEPARTURES:
    for minutes in MAX_TIMES:
        label = max_time_label(minutes)
        key = f"{departure}_{label}"

        summary_path = OUTPUTS_DIR / f"accessibility_summary_3000m_departure_{departure}_max_{label}.parquet"
        od_path = OUTPUTS_DIR / f"od_pairs_3000m_departure_{departure}_max_{label}.parquet"

        summary = gpd.read_parquet(summary_path).to_crs("EPSG:4326")
        count_col = f"n_destinations_reachable_{label}"
        if count_col in summary.columns:
            summary = summary.rename(columns={count_col: "n_destinations_reachable"})
        else:
            summary["n_destinations_reachable"] = 0

        if "from_id" not in summary.columns:
            summary["from_id"] = summary["id"].astype(str)
        summary["from_id"] = summary["from_id"].fillna(summary["id"].astype(str)).astype(str)
        summary["id"] = summary["id"].astype(str)
        summary["departure_label"] = departure
        summary["max_time_minutes"] = minutes

        keep_cols = [
            "id",
            "GDENAME",
            "GDENR",
            "KTNAME",
            "KTKZ",
            "nearest_transit_stop_m",
            "from_id",
            "min_travel_time_to_ic",
            "n_destinations_reachable",
            "departure_label",
            "max_time_minutes",
            "geometry",
        ]
        keep_cols = [col for col in keep_cols if col in summary.columns]
        accessibility_file = f"accessibility_{departure}_{label}.geojson"
        summary[keep_cols].to_file(PTA_IC_DOCS_DIR / accessibility_file, driver="GeoJSON")

        od = pd.read_parquet(od_path)
        od["from_id"] = od["from_id"].astype(str)
        od["to_id"] = od["to_id"].astype(str)
        od = od.merge(ic_lookup, on="to_id", how="left")
        od = od.sort_values(["from_id", "travel_time", "stop_name"], na_position="last")

        reachable = {}
        for origin_id, group in od.groupby("from_id", sort=False):
            destinations = group[["to_id", "stop_name", "travel_time", "lat", "lon"]].to_dict(orient="records")
            reachable[str(origin_id)] = {
                "origin_id": str(origin_id),
                "departure_label": departure,
                "max_time_minutes": minutes,
                "destinations": destinations,
            }

        reachable_file = f"reachable_{departure}_{label}.json"
        with open(PTA_IC_DOCS_DIR / reachable_file, "w", encoding="utf-8") as file:
            json.dump(reachable, file, ensure_ascii=False, separators=(",", ":"))

        manifest["files"][key] = {
            "departure_label": departure,
            "max_time_minutes": minutes,
            "accessibility": f"ptA_IC/{accessibility_file}",
            "reachable": f"ptA_IC/{reachable_file}",
            "feature_count": int(len(summary)),
            "reachable_origin_count": int(len(reachable)),
            "od_pair_count": int(len(od)),
        }

ic_hubs.to_file(PTA_IC_DOCS_DIR / "ic_hubs.geojson", driver="GeoJSON")
manifest["ic_hubs"] = "ptA_IC/ic_hubs.geojson"

with open(PTA_IC_DOCS_DIR / "manifest.json", "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2, ensure_ascii=False)

print(f"Converted {len(manifest['files'])} ptA_IC combinations into {PTA_IC_DOCS_DIR}.")

Converted 32 ptA_IC combinations into docs/data/ptA_IC.


In [17]:
manifest_path = PTA_IC_DOCS_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

missing_web_files = []
for key, entry in manifest["files"].items():
    for field in ["accessibility", "reachable"]:
        path = DOCS_DATA_DIR / entry[field]
        if not path.exists():
            missing_web_files.append(path)

if missing_web_files:
    raise FileNotFoundError(f"Missing generated web files: {missing_web_files[:5]}")

sample_key = "0900_030min"
sample_entry = manifest["files"][sample_key]
sample_geojson = json.loads((DOCS_DATA_DIR / sample_entry["accessibility"]).read_text(encoding="utf-8"))
sample_reachable = json.loads((DOCS_DATA_DIR / sample_entry["reachable"]).read_text(encoding="utf-8"))

print("Manifest combinations:", len(manifest["files"]))
print("Sample feature count:", len(sample_geojson["features"]))
print("Sample reachable origins:", len(sample_reachable))

Manifest combinations: 32
Sample feature count: 4626
Sample reachable origins: 113


## PtA_PtB Conversion

In [18]:
manifest_file = PTA_PTB_OUTPUTS_DIR / "manifest.parquet"
index_file = PTA_PTB_OUTPUTS_DIR / "origin_chunk_index.parquet"

if not manifest_file.exists() or not index_file.exists():
    print("ptA_ptB outputs are not present yet.")
else:
    ptb_manifest = pd.read_parquet(manifest_file)
    ptb_index = pd.read_parquet(index_file)
    chunk_files = ptb_manifest["file"].map(resolve_ptb_chunk_file)
    missing_chunks = [path for path in chunk_files if not path.exists()]
    if missing_chunks:
        raise FileNotFoundError(f"Missing ptA_ptB chunk files: {missing_chunks[:5]}")

    print("ptA_ptB manifest rows:", len(ptb_manifest))
    print("ptA_ptB index rows:", len(ptb_index))
    print("Chunks by departure:")
    print(ptb_manifest.groupby("departure_label").size())

ptA_ptB manifest rows: 76
ptA_ptB index rows: 18504
Chunks by departure:
departure_label
0900    19
1200    19
1700    19
2200    19
dtype: int64


In [22]:
PTA_PTB_DOCS_DIR = DOCS_DATA_DIR / "ptA_ptB"
PTA_PTB_DOCS_DIR.mkdir(parents=True, exist_ok=True)

ptb_manifest_file = PTA_PTB_OUTPUTS_DIR / "manifest.parquet"
ptb_index_file = PTA_PTB_OUTPUTS_DIR / "origin_chunk_index.parquet"

if not ptb_manifest_file.exists() or not ptb_index_file.exists():
    raise FileNotFoundError("Missing outputs/outputs_ptA_ptB manifest files. Run ptA_ptB_calc.ipynb first.")

ptb_manifest = pd.read_parquet(ptb_manifest_file)
ptb_origin_index = pd.read_parquet(ptb_index_file)

# Build municipality lookup from grid points
SWISS_CRS = "EPSG:2056"
municipality_boundaries = gpd.read_file(
    "data/Boundaries_K4_Commune_20260101.gpkg",
    layer="boundaries",
).to_crs(SWISS_CRS)

grid_points = gpd.read_parquet("outputs/outputs_nearest_ic/grid_points_3000m.parquet")
if grid_points.crs is None:
    grid_points = grid_points.set_crs("EPSG:4326")

muni_cols_to_drop = ["GDENAME", "KTNAME", "KTKZ", "GDEHISTID", "GDENR", "KTNR"]
grid_points = grid_points.drop(
    columns=[c for c in muni_cols_to_drop if c in grid_points.columns],
    errors="ignore",
)

grid_points_with_muni = gpd.sjoin(
    grid_points.to_crs(SWISS_CRS),
    municipality_boundaries[["GDENAME", "KTNAME", "KTKZ", "geometry"]],
    how="left",
    predicate="within",
).drop(columns=["index_right"], errors="ignore").drop_duplicates(subset=["id"])

grid_points_with_muni["id"] = grid_points_with_muni["id"].astype(str)
muni_lookup = grid_points_with_muni[["id", "GDENAME", "KTNAME", "KTKZ"]].set_index("id")

ptb_web_manifest = {
    "departures": [
        {"label": f"{departure[:2]}:{departure[2:]}", "value": departure}
        for departure in sorted(ptb_manifest["departure_label"].astype(str).unique())
    ],
    "max_time_minutes": 240,
    "chunks": {},
    "origin_index": {},
}

def safe_str(val):
    if val is None:
        return None
    try:
        if pd.isna(val):
            return None
    except (TypeError, ValueError):
        pass
    return str(val)

for _, row in ptb_manifest.iterrows():
    departure = str(row["departure_label"])
    start = int(row["origin_chunk_start"])
    end = int(row["origin_chunk_end"])
    key = f"{departure}_{start:05d}_{end:05d}"
    source_file = resolve_ptb_chunk_file(row["file"])
    output_file = PTA_PTB_DOCS_DIR / f"travel_times_{key}.json"

    matrix = pd.read_parquet(source_file, columns=["from_id", "to_id", "travel_time"])
    matrix = matrix.dropna(subset=["travel_time"]).copy()
    matrix["from_id"] = matrix["from_id"].astype(str)
    matrix["to_id"] = matrix["to_id"].astype(str)
    matrix["travel_time"] = matrix["travel_time"].round().astype(int)

    payload = {}
    for origin_id, group in matrix.groupby("from_id", sort=False):
        muni = muni_lookup.loc[origin_id] if origin_id in muni_lookup.index else None
        payload[str(origin_id)] = {
            "GDENAME": safe_str(None if muni is None else muni["GDENAME"]),
            "KTNAME":  safe_str(None if muni is None else muni["KTNAME"]),
            "KTKZ":    safe_str(None if muni is None else muni["KTKZ"]),
            "destinations": dict(zip(group["to_id"], group["travel_time"])),
        }

    with open(output_file, "w", encoding="utf-8") as file:
        json.dump(payload, file, separators=(",", ":"))

    ptb_web_manifest["chunks"][key] = {
        "departure_label": departure,
        "origin_chunk_start": start,
        "origin_chunk_end": end,
        "file": f"ptA_ptB/{output_file.name}",
        "origin_count": len(payload),
        "reachable_pair_count": int(len(matrix)),
    }

for _, row in ptb_origin_index.iterrows():
    departure = str(row["departure_label"])
    from_id = str(row["from_id"])
    start = int(row["origin_chunk_start"])
    end = int(row["origin_chunk_end"])
    key = f"{departure}_{start:05d}_{end:05d}"
    ptb_web_manifest["origin_index"].setdefault(departure, {})[from_id] = key

with open(PTA_PTB_DOCS_DIR / "manifest.json", "w", encoding="utf-8") as file:
    json.dump(ptb_web_manifest, file, separators=(",", ":"))

print("Converted ptA_ptB chunks:", len(ptb_web_manifest["chunks"]))

Converted ptA_ptB chunks: 76


In [25]:
ptb_web_manifest = json.loads((PTA_PTB_DOCS_DIR / "manifest.json").read_text(encoding="utf-8"))
missing_ptb_files = []
for key, entry in ptb_web_manifest["chunks"].items():
    path = DOCS_DATA_DIR / entry["file"]
    if not path.exists() or path.stat().st_size == 0:
        missing_ptb_files.append(path)

if missing_ptb_files:
    raise FileNotFoundError(f"Missing generated ptA_ptB web files: {missing_ptb_files[:5]}")

sample_departure = "0900"
sample_origin = next(iter(ptb_web_manifest["origin_index"][sample_departure]))
sample_chunk_key = ptb_web_manifest["origin_index"][sample_departure][sample_origin]
sample_chunk_file = DOCS_DATA_DIR / ptb_web_manifest["chunks"][sample_chunk_key]["file"]
sample_chunk = json.loads(sample_chunk_file.read_text(encoding="utf-8"))

print("ptA_ptB web chunks:", len(ptb_web_manifest["chunks"]))
print("Sample origin:", sample_origin)
print("Reachable destinations from sample origin:", len(sample_chunk.get(sample_origin, {})))


ptA_ptB web chunks: 76
Sample origin: 1000
Reachable destinations from sample origin: 4


In [26]:
# Fix grid_points_3000m.parquet — strip bad municipality columns and re-attach correctly
import geopandas as gpd

SWISS_CRS = "EPSG:2056"
municipality_boundaries = gpd.read_file(
    "data/Boundaries_K4_Commune_20260101.gpkg",
    layer="boundaries",
).to_crs(SWISS_CRS)

grid_points = gpd.read_parquet("outputs/outputs_nearest_ic/grid_points_3000m.parquet")
if grid_points.crs is None:
    grid_points = grid_points.set_crs("EPSG:4326")

# Drop stale municipality columns
muni_cols = ["GDENAME", "KTNAME", "KTKZ", "GDEHISTID", "GDENR", "KTNR"]
grid_points = grid_points.drop(
    columns=[c for c in muni_cols if c in grid_points.columns],
    errors="ignore",
)

# Re-attach correctly via spatial join
grid_points_fixed = gpd.sjoin(
    grid_points.to_crs(SWISS_CRS),
    municipality_boundaries[["GDENAME", "KTNAME", "KTKZ", "geometry"]],
    how="left",
    predicate="within",
).drop(columns=["index_right"], errors="ignore").drop_duplicates(subset=["id"])

grid_points_fixed.to_parquet("outputs/outputs_nearest_ic/grid_points_3000m.parquet", index=False)

# Verify
print(grid_points_fixed[["id", "GDENAME", "KTNAME"]].head(10))
print("Unique GDENAME count:", grid_points_fixed["GDENAME"].nunique())

    id   GDENAME  KTNAME
0   11    Chancy  Genève
1   12    Chancy  Genève
2   13  Dardagny  Genève
3   14  Dardagny  Genève
4   85     Avusy  Genève
5   86     Avusy  Genève
6   87    Russin  Genève
7   88   Satigny  Genève
8   89   Satigny  Genève
9  159     Soral  Genève
Unique GDENAME count: 1589
